# Get arabic wiki files


In [ ]:
import kagglehub  # Import the kagglehub library for interacting with Kaggle datasets

# Download latest version
path = kagglehub.dataset_download("z3rocool/arabic-wikipedia-dump-2021")  # Download the dataset "arabic-wikipedia-dump-2021" from Kaggle

print("Path to dataset files:", path)  # Print the path where the dataset files are stored


In [ ]:
import shutil  # Import the shutil library for file operations

# copy files from cache

source_dir = path  # Set the source directory (path to the dataset files)
destination_dir = '/content/new_dir'  # Set the destination directory for copying files

# Create the destination directory if it doesn't exist
# !mkdir -p /content/new_dir  # Uncomment this line if you want to create the destination directory manually

shutil.copytree(source_dir, destination_dir)  # Copy the entire directory from source to destination
print("Directory copied successfully!")  # Print confirmation message

!ls /content/new_dir  # List the contents of the destination directory to verify the copy


In [ ]:
!tar -xf /content/new_dir/ar_corpus.tar.xz -C /content/new_dir
# unzip tar file

In [ ]:
import os

len(os.listdir('/content/new_dir/ar_corpus'))

# Convert raw data to structured

In [ ]:
import os
from tqdm import tqdm

def process_directory(input_dir, output_dir):
    """
    Process all text files in the input directory, reformat them, and save them in the output directory.

    :param input_dir: Path to the directory containing text files.
    :param output_dir: Path to the directory to save reformatted files.
    """
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Collect all text files from the directory
    text_files = []
    for root, _, files in os.walk(input_dir):
        for file_name in files:
            if file_name.endswith('.txt'):
                text_files.append(os.path.join(root, file_name))


    # Process each text file with a progress bar
    for input_file_path in tqdm(text_files, desc="Processing files", unit="file"):
        relative_path = os.path.relpath(os.path.dirname(input_file_path), input_dir)
        output_file_path = os.path.join(output_dir, relative_path, os.path.basename(input_file_path))

        # Ensure subdirectories in the output directory exist
        os.makedirs(os.path.dirname(output_file_path), exist_ok=True)

        try:
            with open(input_file_path, 'r', encoding='utf-8') as file:
                # Process the file content
                new_format = file.readline().replace('=====', '\n 5 \n')
                new_format = new_format.replace('====', '\n 4 \n')
                new_format = new_format.replace('===', '\n 3 \n')
                new_format = new_format.replace('==', '\n 2 \n')
                new_format = new_format.split('\n')

                data = "========,1,مقدمة \n"
                for i, item in enumerate(new_format):
                    if (i < len(new_format) - 2) and (
                        item in [' 5 ', ' 4 ', ' 3 ', ' 2 ']
                    ):
                        if item == new_format[i + 2]:
                            data += f"========,{item.strip()},{new_format[i + 1]} \n"
                            new_format[i + 2] = ""
                            new_format[i + 1] = ""
                        else:
                            continue
                    else:
                        if item.strip() == '':
                            continue
                        data += item + '\n'

            # Save the reformatted data to the output directory
            with open(output_file_path, 'w', encoding='utf-8') as output_file:
                output_file.write(data)

        except FileNotFoundError:
            print(f"Error: File not found - {input_file_path}")
        except Exception as e:
            print(f"An error occurred while processing {input_file_path}: {e}")

input_directory = '/content/new_dir/ar_corpus'
output_directory = '/content/reformat_data'

process_directory(input_directory, output_directory)


##Articles Cleaning


In [ ]:
import os
import re
from tqdm import tqdm

def clean_text_file(input_file, output_file, sections_to_remove):
    """
    Cleans a single text file by removing specified sections, ensuring each sentence ends with a newline, and removing empty lines.

    :param input_file: Path to the input text file.
    :param output_file: Path to save the cleaned text file.
    :param sections_to_remove: List of section names to remove.
    """
    with open(input_file, 'r', encoding='utf-8') as infile:
        lines = infile.readlines()

    cleaned_lines = []
    skip_section = False  # Flag to track whether we're skipping a section

    for line in lines:
        line = line.strip()  # Remove leading/trailing spaces

        # Check if the line indicates a section header
        if line.startswith("========") and any(section in line for section in sections_to_remove):
            skip_section = True  # Start skipping this section
            continue

        # End skipping when we encounter the next section header
        if line.startswith("========") and skip_section:
            skip_section = False
            continue

        # Skip lines within the unwanted section
        if skip_section:
            continue

        if not line:
            continue

        # Ensure each sentence ends with a newline after "."
        line = re.sub(r'(?<!\n)\.', '.\n', line)
        if len(line)<10:
          continue
        cleaned_lines.append(line.strip())

    # Write cleaned content to the output file
    cleaned_text = '\n'.join(cleaned_lines)
    m = 1000
    clean_string = cleaned_text.split('\n')
    cleaned_lines = []
    for i in clean_string:
      if len(i) < 10:
        continue
      cleaned_lines.append(i)

    with open(output_file, 'w', encoding='utf-8') as outfile:
        outfile.write("\n".join(cleaned_lines))
    # with open(output_file, 'w', encoding='utf-8') as outfile:
    #     outfile.write()

def clean_text_files_in_directory(input_dir, output_dir, sections_to_remove):
    """
    Processes all text files in the input directory and saves cleaned versions to the output directory.

    :param input_dir: Directory containing input text files.
    :param output_dir: Directory to save cleaned text files.
    :param sections_to_remove: List of section names to remove.
    """
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Get a list of all text files in the directory
    text_files = []
    for root, _, files in os.walk(input_dir):
        for file_name in files:
            if file_name.endswith('.txt'):  # Process only .txt files
                input_file_path = os.path.join(root, file_name)
                relative_path = os.path.relpath(root, input_dir)
                output_file_dir = os.path.join(output_dir, relative_path)
                output_file_path = os.path.join(output_file_dir, file_name)
                text_files.append((input_file_path, output_file_path, output_file_dir))

    # Process files with tqdm
    for input_file, output_file, output_file_dir in tqdm(text_files, desc="Processing files", unit="file"):
        os.makedirs(output_file_dir, exist_ok=True)  # Ensure subdirectories exist
        clean_text_file(input_file, output_file, sections_to_remove)

input_directory = '/content/reformat_data'  # Directory containing input text files
output_directory = '/content/output'  # Directory to save cleaned text files
sections_to_remove = ["انظر أيضًا", "مراجع", "مصادر"]  # Sections to remove

clean_text_files_in_directory(input_directory, output_directory, sections_to_remove)

In [ ]:
len(os.listdir('/content/output'))

In [ ]:
file_path = '/content/output/article_271458.txt'  # Specify the file path to read

with open(file_path, 'r', encoding='utf-8') as file:  # Open the file in read mode with UTF-8 encoding
        content = file.read()  # Read the entire content of the file
        print(content)  # Print the content of the file to the console


## Deleting Small files

In [ ]:
import os
# delete all filse that's contain at most 3 lines
def delete_small_files(directory, min_lines=3):
    """
    Delete files in the specified directory that contain fewer than `min_lines` lines.

    :param directory: Path to the directory containing text files.
    :param min_lines: Minimum number of lines a file must have to avoid deletion.
    """
    # Iterate through all files in the directory
    for filename in os.listdir(directory):
        file_path = os.path.join(directory, filename)

        # Ensure we process only text files
        if os.path.isfile(file_path) and filename.endswith('.txt'):
            with open(file_path, 'r', encoding='utf-8') as file:
                # Count the lines in the file
                lines = file.readlines()
                if len(lines)<= min_lines:
                    # Delete the file if it has fewer than `min_lines`
                    os.remove(file_path)
                    print(f"Deleted {filename}: contains {len(lines)} lines.")

input_directory = '/content/output'
delete_small_files(input_directory, min_lines=3)


## Data Processing

In [ ]:
# download lib

import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab') # Download the punkt_tab data package
!pip install pyarabic

In [ ]:
import os
import re
from tqdm import tqdm
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from pyarabic.araby import sentence_tokenize as Pasent_tokenize


def clean_arabic_text(text):
    # Remove extra spaces and newlines
    text = re.sub(r'\s+', ' ', text).strip()

    # Normalize Arabic characters
    text = re.sub(r'[إأآا]', 'ا', text)  # Normalize Alef
    text = re.sub(r'ئ', 'ي', text)  # Normalize Ya
    text = re.sub(r'ؤ', 'و', text)  # Normalize Waw
    text = re.sub(r'ة', 'ه', text)  # Replace Ta Marbuta with Ha

    # Remove diacritics (Tashkeel)
    arabic_diacritics = re.compile(r'[\u0617-\u061A\u064B-\u0652]')
    text = re.sub(arabic_diacritics, '', text)

    # Remove non-Arabic characters (e.g., English, numbers, and symbols)
    arabic_chars_and_punctuation = r'[^ء-ي\s.,,.!?؟؛]'
    text = re.sub(arabic_chars_and_punctuation, '', text)

    # Remove meaningless or standalone "و"
    text = re.sub(r'\bو\s+', '', text)  # Remove "و" followed by spaces
    text = re.sub(r'\s+و\s+', ' ', text)  # Remove redundant spaces around "و"

    tokens = word_tokenize(text)  # Tokenize
    # removing stop words
    stop_words = set(stopwords.words('arabic'))
    tokens = [word for word in tokens if word not in stop_words]
    line_after_clean = " ".join(tokens)
    sentences = Pasent_tokenize(line_after_clean)

    for i in range(len(sentences)-1,0,-1):
      if '=======' not in sentences[i] and len(sentences[i].split())<20:
        sentences.pop(i)

    return sentences

def clean_text_files_in_directory(input_dir, output_dir):
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Get the list of files in the input directory
    files = [f for f in os.listdir(input_dir) if f.endswith('.txt')]
    files.sort()


    # Iterate through the selected files with tqdm for progress tracking
    for filename in tqdm(files, desc="Processing files"):
        input_file_path = os.path.join(input_dir, filename)
        output_file_path = os.path.join(output_dir, filename)

        with open(input_file_path, 'r', encoding='utf-8') as infile:
            lines = infile.readlines()
        cleaned_lines = []
        for line in lines:
            # If the line starts with "========", keep it unchanged
            if line.startswith('========'):

                cleaned_lines.append(line.strip())
            # If it's the "id,title" line, keep it unchanged
            elif re.match(r'^\d+,.+$', line.strip()):
                cleaned_lines.append(line.strip())
            else:
                # Clean the Arabic text
                cleaned_line = clean_arabic_text(line)
                if cleaned_line:  # Only add non-empty lines
                    cleaned_lines.extend(cleaned_line)

        # Remove multiple empty lines (ensure single empty line)
        result = []
        for i, line in enumerate(cleaned_lines):
            if line == '' and (i == 0 or cleaned_lines[i - 1] == ''):
                continue
            result.append(line)

        # Write the cleaned content to the output file
        with open(output_file_path, 'w', encoding='utf-8') as outfile:
            outfile.write('\n'.join(result) + '\n')


input_directory = '/content/output'
output_directory = '/content/cleaned_data'  # Directory to save cleaned text files

clean_text_files_in_directory(input_directory, output_directory)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Saving the data

In [ ]:
import shutil  # Import the shutil module for file operations

input_folder = "/content/cleaned_data"  # Set the path to the folder to be archived
output_zip = "/content/drive/MyDrive/SegmentedArticles_cleaned_Zipped"  # Set the output path for the zip file

shutil.make_archive(output_zip, 'zip', input_folder)  # Create a zip archive of the input folder and save it to the output path


In [ ]:
import os
all_files = ['/content/cleaned_data/'+ x for x in os.listdir('/content/cleaned_data') if ".txt" in x]

## Saving the files in a tree structre the same as the original Wiki dataset

In [ ]:
import os
import shutil
import itertools
import string
from tqdm import tqdm  # Import tqdm for progress tracking


# Define the main dataset directory
main_folder = "/content/cleaned_data"

# Ensure the main folder exists
os.makedirs(main_folder, exist_ok=True)

# Generating 99 subfolder names (AA, AB, ..., ZZ)
subfolder_names = [''.join(x) for x in itertools.islice(itertools.product(string.ascii_uppercase, repeat=2), 99)]

# Total number of files
total_files = len(all_files)
file_index = 0  # Tracking the file position
all_subfolders = []  # Storing all available subfolders

# Step 1: Create the entire folder structure first
for subfolder in subfolder_names:
    subfolder_path = os.path.join(main_folder, subfolder)
    os.makedirs(subfolder_path, exist_ok=True)

    for sub_subfolder_num in range(100):  # Create 100 subfolders inside each subfolder
        sub_subfolder_path = os.path.join(subfolder_path, f"{sub_subfolder_num:03d}")
        os.makedirs(sub_subfolder_path, exist_ok=True)
        all_subfolders.append(sub_subfolder_path)  # Store for later assignment

# Step 2: Distribute the files evenly across all subfolders
total_subfolders = len(all_subfolders)
files_per_folder = total_files // total_subfolders  # Standard distribution
extra_files = total_files % total_subfolders  # Handle remainder files

# Progress bar
with tqdm(total=total_files, desc="Distributing Files", unit="file") as pbar:
    for subfolder_path in all_subfolders:
        num_files_to_move = files_per_folder + (1 if extra_files > 0 else 0)
        extra_files -= 1  # Reduce the extra file count

        for _ in range(num_files_to_move):
            if file_index < total_files:
                file_path = all_files[file_index]
                file_name = os.path.basename(file_path)
                destination = os.path.join(subfolder_path, file_name)

                shutil.move(file_path, destination)  # Move the file to the target location
                file_index += 1
                pbar.update(1)  # Update progress bar
            else:
                break  # Stop if all files are assigned

print("Dataset organization complete!")


In [ ]:
import shutil

destination_zip = "/content/drive/MyDrive/ar_wiki727.zip"

# Create the zip archive
shutil.make_archive(destination_zip.replace(".zip", ""), 'zip', source_dir)

print(f"Zipped {main_folder } to {destination_zip} successfully!")
